# Neuromorphic SNN vs ANN on CIFAR-10

**Ethan Vu, Marcus Vu** - CMPE 257

We train a regular CNN and a spiking neural network (SNN) with the same layers on CIFAR-10 and compare accuracy, training time, inference speed, GPU usage and energy.
The SNN uses Leaky Integrate-and-Fire neurons, surrogate gradients and Temporal Effective Batch Normalization (TEBN) from
[Duan et al., NeurIPS 2022](https://proceedings.neurips.cc/paper_files/paper/2022/hash/de2ad3ed44ee4e675b3be42aa0b615d0-Abstract-Conference.html).

**Running on Kaggle:** Settings -> Accelerator -> GPU T4 (one GPU), and turn Internet on (needed to download CIFAR-10).
Set `QUICK_RUN = True` first to make sure everything runs, then set it back to `False`.

**Local GPU:** install a compatible CUDA-enabled PyTorch build in the Jupyter kernel. Set `SNN_DEVICE=cuda:0` (or another visible index) to require that GPU, or `cpu` to force CPU. `auto` is the default. `SNN_BATCH_SIZE`, `SNN_NUM_WORKERS` and `SNN_QUICK_RUN=1` can be set before starting Jupyter. See the local setup in README.md.


In [ ]:
!pip install -q nvidia-ml-py

## Setup

In [ ]:
import os
import time
import threading

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


def select_device(requested="auto"):
    """Select one local CPU/CUDA device and verify CUDA kernels before training."""
    if requested == "auto":
        requested = "cuda:0" if torch.cuda.is_available() else "cpu"
    selected = torch.device(requested)
    if selected.type not in ("cpu", "cuda"):
        raise ValueError("SNN_DEVICE must be auto, cpu or cuda:N")
    if selected.type == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable. Install compatible CUDA-enabled "
                               "PyTorch/torchvision in this notebook kernel; see README.md.")
        index = selected.index if selected.index is not None else torch.cuda.current_device()
        if not 0 <= index < torch.cuda.device_count():
            raise ValueError(f"CUDA device {index} is not visible to PyTorch")
        selected = torch.device("cuda", index)
        try:
            torch.cuda.set_device(selected)
            (torch.ones(1, device=selected) * 2).item()
            torch.cuda.synchronize(selected)
        except RuntimeError as error:
            raise RuntimeError("CUDA startup failed. Check the driver and PyTorch GPU "
                               "architecture support, or set SNN_DEVICE=cpu; see README.md.") from error
    return selected


device = select_device(os.environ.get("SNN_DEVICE", "auto"))
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
    print("GPU capability:", torch.cuda.get_device_capability(device))
    print("Build architectures:", torch.cuda.get_arch_list())

# Never sample another GPU (or sample GPU power for a CPU run).
HAS_POWER_READINGS = False
if device.type == "cuda":
    try:
        import pynvml
        pynvml.nvmlInit()
        gpu_uuid = str(torch.cuda.get_device_properties(device).uuid)
        gpu_handle = pynvml.nvmlDeviceGetHandleByUUID(gpu_uuid)
        pynvml.nvmlDeviceGetPowerUsage(gpu_handle)  # verify this GPU supports power readings
        HAS_POWER_READINGS = True
    except Exception as error:
        print("GPU power unavailable:", error)
print("GPU power readings available:", HAS_POWER_READINGS)


In [ ]:
QUICK_RUN = os.environ.get("SNN_QUICK_RUN", "0") == "1"        # True = small subset + 2 epochs, just to check the code runs

BATCH_SIZE = int(os.environ.get("SNN_BATCH_SIZE", "128"))
NUM_WORKERS = int(os.environ.get("SNN_NUM_WORKERS", "0" if os.name == "nt" else "2"))
if BATCH_SIZE < 1 or NUM_WORKERS < 0:
    raise ValueError("SNN_BATCH_SIZE must be positive and SNN_NUM_WORKERS nonnegative")
EPOCHS = 30              # the paper uses 300; 30 is enough to compare the two models
LEARNING_RATE = 0.05
WEIGHT_DECAY = 5e-4

TIMESTEPS = 4            # how many time steps the SNN runs for each image
DECAY = 0.5              # how much membrane potential is kept between time steps
THRESHOLD = 1.0          # membrane potential needed to fire a spike

# Energy per operation on 45nm chips (Horowitz, 2014) - the standard numbers in SNN papers
ENERGY_PER_MAC = 4.6e-12   # joules, multiply-accumulate (what ANNs do)
ENERGY_PER_AC = 0.9e-12    # joules, accumulate only (what SNNs do when a spike arrives)

OUTPUT_DIR = "outputs"     # results, plots and model checkpoints are saved here
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

## Data

CIFAR-10 has 60,000 32x32 color images in 10 classes (50k train / 10k test).
Images are turned into tensors and normalized. Training images also get random crops, flips, color jitter and small rotations.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
CLASS_NAMES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

# Training images get random augmentations; test images are only normalized.
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.2, 0.2, 0.2, 0.05)], p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_data = datasets.CIFAR10(root="data", train=True, download=True, transform=train_transform)
test_data = datasets.CIFAR10(root="data", train=False, download=True, transform=test_transform)

if QUICK_RUN:
    train_data = Subset(train_data, range(2000))
    test_data = Subset(test_data, range(1000))
    EPOCHS = 2

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=NUM_WORKERS)

print(f"{len(train_data)} training images, {len(test_data)} test images")

In [ ]:
images, labels = next(iter(train_loader))
mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
std = torch.tensor(CIFAR_STD).view(3, 1, 1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, image, label in zip(axes.flat, images, labels):
    image = (image * std + mean).clamp(0, 1)          # undo normalization for display
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[label])
    ax.axis("off")
plt.show()

## ANN baseline

A small VGG-style CNN: six 3x3 conv layers (conv -> batch norm -> ReLU), three max pools and two linear layers.
The SNN below uses exactly the same layers so the comparison is fair.

In [ ]:
def conv_bn_relu(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
    )


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            conv_bn_relu(3, 64),    conv_bn_relu(64, 64),   nn.MaxPool2d(2),   # 32x32 -> 16x16
            conv_bn_relu(64, 128),  conv_bn_relu(128, 128), nn.MaxPool2d(2),   # 16x16 -> 8x8
            conv_bn_relu(128, 256), conv_bn_relu(256, 256), nn.MaxPool2d(2),   # 8x8   -> 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, images):
        return self.classifier(self.features(images))

## Spiking neural network

### Spike function
A neuron outputs 1 when its membrane potential crosses the threshold and 0 otherwise. That step has no useful gradient,
so in the backward pass we use a triangle-shaped surrogate gradient instead.

In [ ]:
class SpikeFunction(torch.autograd.Function):
    """
    Forward:  fire a spike (1) if the membrane potential is above threshold, else 0.
    Backward: a step function has zero gradient everywhere, so training would stall.
              We pretend its gradient is a triangle around the threshold instead.
    """

    @staticmethod
    def forward(ctx, distance_above_threshold):
        ctx.save_for_backward(distance_above_threshold)
        return (distance_above_threshold >= 0).float()

    @staticmethod
    def backward(ctx, grad_output):
        (distance_above_threshold,) = ctx.saved_tensors
        triangle = torch.clamp(1.0 - distance_above_threshold.abs(), min=0)
        return grad_output * triangle

### LIF neuron
Each time step the membrane potential leaks (`u = DECAY * u`), adds the input, fires if it reaches `THRESHOLD`, and resets to 0 after firing.

In [ ]:
class LIFNeuron(nn.Module):
    """
    Input and output shape: [batch, time, ...]
    At each time step the neuron:
      1. leaks:      keeps DECAY of its old membrane potential
      2. integrates: adds the new input
      3. fires:      outputs a spike if potential >= THRESHOLD
      4. resets:     potential goes back to 0 after a spike
    """

    def __init__(self):
        super().__init__()
        self.recording = False      # when True, we store the firing rate of each batch
        self.firing_rates = []

    def forward(self, inputs):
        membrane = torch.zeros_like(inputs[:, 0])
        spikes = []

        for t in range(inputs.shape[1]):
            membrane = DECAY * membrane + inputs[:, t]
            spike = SpikeFunction.apply(membrane - THRESHOLD)
            membrane = membrane * (1 - spike)        # reset neurons that fired
            spikes.append(spike)

        spikes = torch.stack(spikes, dim=1)

        if self.recording:
            self.firing_rates.append(spikes.mean().item())
        return spikes

### Running normal layers over time
SNN tensors have an extra time dimension: `[batch, time, channels, height, width]`. This wrapper lets regular PyTorch layers handle that.

In [ ]:
class EveryTimestep(nn.Module):
    """
    Conv, Linear and MaxPool layers don't know about time.
    We merge the batch and time dimensions, run the layer once, then split them again.
    """

    def __init__(self, layer):
        super().__init__()
        self.layer = layer

    def forward(self, x):
        batch, time_steps = x.shape[0], x.shape[1]
        out = self.layer(x.flatten(0, 1))                  # [batch * time, ...]
        return out.view(batch, time_steps, *out.shape[1:])  # [batch, time, ...]

### TEBN
Normal batch norm doesn't account for time. TEBN computes the statistics over the batch and all time steps together,
then multiplies each time step by its own learned scale `p[t]`.

In [ ]:
class TEBN(nn.Module):
    """
    TEBN from Duan et al. (NeurIPS 2022):
      1. Normalize using mean/variance computed over the batch AND all time steps together.
      2. Multiply each time step by its own learnable scale p[t].
    """

    def __init__(self, channels):
        super().__init__()
        self.bn = nn.BatchNorm2d(channels)
        self.timestep_scale = nn.Parameter(torch.ones(1, TIMESTEPS, 1, 1, 1))

    def forward(self, x):                                   # x: [batch, time, C, H, W]
        normalized = self.bn(x.flatten(0, 1)).view_as(x)    # stats over batch and time
        return normalized * self.timestep_scale


class PlainBN(nn.Module):
    """Ordinary batch norm with no time scaling, used only for the optional comparison at the end."""

    def __init__(self, channels):
        super().__init__()
        self.bn = nn.BatchNorm2d(channels)

    def forward(self, x):
        return self.bn(x.flatten(0, 1)).view_as(x)

### SNN model
Same layers as the CNN, with TEBN instead of batch norm and LIF neurons instead of ReLU.
The image is fed in at every time step (direct encoding) and the outputs are averaged over time.

In [ ]:
def conv_norm_lif(in_channels, out_channels, norm_layer):
    return nn.Sequential(
        EveryTimestep(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)),
        norm_layer(out_channels),
        LIFNeuron(),
    )


def max_pool():
    return EveryTimestep(nn.MaxPool2d(2))


class SNN(nn.Module):
    def __init__(self, use_tebn=True):
        super().__init__()
        norm = TEBN if use_tebn else PlainBN

        self.features = nn.Sequential(
            conv_norm_lif(3, 64, norm),    conv_norm_lif(64, 64, norm),   max_pool(),
            conv_norm_lif(64, 128, norm),  conv_norm_lif(128, 128, norm), max_pool(),
            conv_norm_lif(128, 256, norm), conv_norm_lif(256, 256, norm), max_pool(),
        )
        self.classifier = nn.Sequential(
            EveryTimestep(nn.Sequential(nn.Flatten(), nn.Linear(256 * 4 * 4, 256))),
            LIFNeuron(),
            EveryTimestep(nn.Linear(256, 10)),
        )

    def forward(self, images):
        # Direct encoding: show the same image at every time step.
        x = images.unsqueeze(1).repeat(1, TIMESTEPS, 1, 1, 1)   # [batch, time, 3, 32, 32]
        outputs = self.classifier(self.features(x))             # [batch, time, 10]
        return outputs.mean(dim=1)                              # average over time

Quick check that both models run and the LIF neurons only output 0s and 1s.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print("CNN parameters:", f"{count_parameters(CNN()):,}")
print("SNN parameters:", f"{count_parameters(SNN()):,}")

test_images = torch.randn(4, 3, 32, 32)
print("CNN output shape:", CNN()(test_images).shape)
print("SNN output shape:", SNN()(test_images).shape)

spikes = LIFNeuron()(torch.randn(2, TIMESTEPS, 5) * 2)
print("LIF outputs only 0s and 1s:", set(spikes.unique().tolist()) <= {0.0, 1.0})

## Measuring

We record for both models:
- accuracy on the test set
- training time
- inference speed (images per second and single-image latency)
- peak GPU memory and GPU utilization
- average GPU power and energy (sampled with NVML every 0.1 s)

In [ ]:
class GPUMeter:
    """
    Use with a `with` block. Afterwards it holds:
      seconds, peak_memory_mb, avg_power_w, avg_gpu_util, energy_j
    Power is sampled every 0.1 s in a background thread.
    """

    def __enter__(self):
        self.power_samples = []
        self.util_samples = []
        self.running = True

        if device.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        if HAS_POWER_READINGS:
            self.thread = threading.Thread(target=self.sample_power, daemon=True)
            self.thread.start()

        self.start_time = time.perf_counter()
        return self

    def sample_power(self):
        while self.running:
            self.power_samples.append(pynvml.nvmlDeviceGetPowerUsage(gpu_handle) / 1000)  # mW -> W
            self.util_samples.append(pynvml.nvmlDeviceGetUtilizationRates(gpu_handle).gpu)
            time.sleep(0.1)

    def __exit__(self, *args):
        if device.type == "cuda":
            torch.cuda.synchronize()
        self.seconds = time.perf_counter() - self.start_time

        self.running = False
        if HAS_POWER_READINGS:
            self.thread.join()

        self.peak_memory_mb = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else float("nan")
        self.avg_power_w = np.mean(self.power_samples) if self.power_samples else float("nan")
        self.avg_gpu_util = np.mean(self.util_samples) if self.util_samples else float("nan")
        self.energy_j = self.avg_power_w * self.seconds

Training uses SGD with Nesterov momentum, a cosine learning-rate schedule, and keeps the best epoch.

In [ ]:
loss_function = nn.CrossEntropyLoss()


def train_one_epoch(model, optimizer):
    model.train()
    total_loss, correct, seen = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        seen += len(labels)

    return total_loss / seen, 100 * correct / seen


@torch.no_grad()
def evaluate(model):
    model.eval()
    correct, seen = 0, 0

    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        predictions = model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        seen += len(labels)

    return 100 * correct / seen


def train_model(model, name):
    """Trains the model, keeps the best checkpoint, and returns (history, stats)."""
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9,
                                weight_decay=WEIGHT_DECAY, nesterov=True)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = []
    best_accuracy = 0.0

    with GPUMeter() as meter:
        for epoch in range(1, EPOCHS + 1):
            start = time.perf_counter()
            train_loss, train_acc = train_one_epoch(model, optimizer)
            epoch_seconds = time.perf_counter() - start

            test_acc = evaluate(model)
            scheduler.step()

            history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                            "test_acc": test_acc, "epoch_seconds": epoch_seconds})
            print(f"{name} | epoch {epoch:2d}/{EPOCHS} | loss {train_loss:.3f} | "
                  f"train {train_acc:5.2f}% | test {test_acc:5.2f}% | {epoch_seconds:.1f}s")

            if test_acc > best_accuracy:
                best_accuracy = test_acc
                torch.save(model.state_dict(), f"{OUTPUT_DIR}/{name}.pt")

    model.load_state_dict(torch.load(f"{OUTPUT_DIR}/{name}.pt", map_location=device, weights_only=True))   # go back to the best epoch

    stats = {
        "best_test_acc": best_accuracy,
        "train_minutes": meter.seconds / 60,
        "train_peak_memory_mb": meter.peak_memory_mb,
        "train_avg_power_w": meter.avg_power_w,
        "train_avg_gpu_util": meter.avg_gpu_util,
        "train_energy_kj": meter.energy_j / 1000,
    }
    return pd.DataFrame(history), stats


@torch.no_grad()
def measure_inference(model):
    """Throughput over the whole test set, plus latency for a single image."""
    model.eval()

    images, _ = next(iter(test_loader))
    model(images.to(device))                       # warm-up run

    with GPUMeter() as meter:
        image_count = 0
        for images, _ in test_loader:
            model(images.to(device))
            image_count += len(images)

    one_image = images[:1].to(device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(100):
        model(one_image)
    if device.type == "cuda":
        torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - start) / 100 * 1000

    return {
        "images_per_second": image_count / meter.seconds,
        "latency_ms": latency_ms,
        "inference_peak_memory_mb": meter.peak_memory_mb,
        "inference_avg_power_w": meter.avg_power_w,
        "gpu_energy_per_image_mj": meter.energy_j / image_count * 1000,
    }

## Training

### CNN

In [ ]:
torch.manual_seed(42)
cnn = CNN()
cnn_history, cnn_stats = train_model(cnn, "cnn")
cnn_stats.update(measure_inference(cnn))
cnn_stats

### SNN (TEBN)

The SNN runs every layer `TIMESTEPS` times, so expect it to be slower on a GPU.

In [ ]:
torch.manual_seed(42)
snn = SNN(use_tebn=True)
snn_history, snn_stats = train_model(snn, "snn_tebn")
snn_stats.update(measure_inference(snn))
snn_stats

## Spike activity

The fraction of neurons that fire is what makes SNNs cheap on neuromorphic hardware: a neuron that doesn't spike doesn't cause any work.

In [ ]:
def measure_firing_rates(model):
    neurons = [m for m in model.modules() if isinstance(m, LIFNeuron)]
    for neuron in neurons:
        neuron.recording = True
        neuron.firing_rates = []

    evaluate(model)                                   # one pass over the test set

    rates = [np.mean(neuron.firing_rates) for neuron in neurons]
    for neuron in neurons:
        neuron.recording = False
    return rates


firing_rates = measure_firing_rates(snn)
layer_names = [f"LIF {i + 1}" for i in range(len(firing_rates))]

plt.figure(figsize=(8, 4))
plt.bar(layer_names, [rate * 100 for rate in firing_rates])
plt.ylabel("neurons firing (%)")
plt.title("SNN firing rate per layer")
plt.show()

print(f"Average firing rate: {np.mean(firing_rates) * 100:.2f}%")

In [ ]:
scales = [m.timestep_scale.detach().cpu().flatten().numpy() for m in snn.modules() if isinstance(m, TEBN)]

plt.figure(figsize=(6, 4))
plt.imshow(np.array(scales), aspect="auto")
plt.colorbar(label="p[t]")
plt.xlabel("time step")
plt.ylabel("TEBN layer")
plt.xticks(range(TIMESTEPS))
plt.title("Learned TEBN time-step scales")
plt.show()

## Theoretical energy

A GPU does the same work whether a neuron spikes or not, so measured GPU power will usually favor the CNN.
To estimate the cost on neuromorphic chips we count operations instead, like most SNN papers do:
- CNN: every multiply-accumulate (MAC) costs 4.6 pJ
- SNN: the first layer still does MACs on the pixels; every later layer only does an addition (0.9 pJ) when a spike arrives

In [ ]:
@torch.no_grad()
def count_operations(model, batches=5):
    """
    For every Conv/Linear layer, record:
      macs            - multiply-accumulates for one input (one time step for the SNN)
      input_activity  - fraction of inputs that are non-zero
      spike_input     - True if the inputs are only 0s and 1s (spikes)
    """
    info = {}

    def make_hook(name):
        def hook(layer, inputs, output):
            x = inputs[0]
            if isinstance(layer, nn.Conv2d):
                macs = output[0].numel() * layer.in_channels * layer.kernel_size[0] * layer.kernel_size[1]
            else:
                macs = layer.in_features * layer.out_features

            entry = info.setdefault(name, {"macs": macs, "activity": [], "spike_input": True})
            entry["activity"].append((x != 0).float().mean().item())
            entry["spike_input"] = entry["spike_input"] and bool(((x == 0) | (x == 1)).all())
        return hook

    hooks = [layer.register_forward_hook(make_hook(name))
             for name, layer in model.named_modules() if isinstance(layer, (nn.Conv2d, nn.Linear))]

    model.eval()
    for i, (images, _) in enumerate(test_loader):
        model(images.to(device))
        if i + 1 == batches:
            break

    for hook in hooks:
        hook.remove()

    return pd.DataFrame([{"layer": name, "macs": e["macs"],
                          "input_activity": np.mean(e["activity"]), "spike_input": e["spike_input"]}
                         for name, e in info.items()])


# ANN: every layer does every multiply-accumulate.
cnn_ops = count_operations(cnn)
cnn_energy = cnn_ops["macs"].sum() * ENERGY_PER_MAC

# SNN: layers run once per time step.
#  - The first layer sees real-valued pixels, so it still does full MACs.
#  - Later layers only do an addition when a spike arrives, so multiply by the activity.
snn_ops = count_operations(snn)
snn_ops["mac_ops"] = np.where(snn_ops["spike_input"], 0, snn_ops["macs"] * TIMESTEPS)
snn_ops["ac_ops"] = np.where(snn_ops["spike_input"], snn_ops["macs"] * TIMESTEPS * snn_ops["input_activity"], 0)
snn_energy = (snn_ops["mac_ops"] * ENERGY_PER_MAC + snn_ops["ac_ops"] * ENERGY_PER_AC).sum()

cnn_stats["theoretical_energy_uj"] = cnn_energy * 1e6
snn_stats["theoretical_energy_uj"] = snn_energy * 1e6

print(snn_ops)
print(f"\nCNN: {cnn_energy * 1e6:.1f} uJ per image")
print(f"SNN: {snn_energy * 1e6:.1f} uJ per image")
print(f"SNN uses {cnn_energy / snn_energy:.1f}x less energy (theoretical, neuromorphic hardware)")

## Results

In [ ]:
results = pd.DataFrame({"CNN (ANN)": cnn_stats, f"SNN (TEBN, T={TIMESTEPS})": snn_stats}).round(2)
results.to_csv(f"{OUTPUT_DIR}/ann_vs_snn_results.csv")
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(cnn_history["epoch"], cnn_history["test_acc"], label="CNN")
axes[0].plot(snn_history["epoch"], snn_history["test_acc"], label="SNN (TEBN)")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("test accuracy (%)")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(cnn_history["epoch"], cnn_history["train_loss"], label="CNN")
axes[1].plot(snn_history["epoch"], snn_history["train_loss"], label="SNN (TEBN)")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("training loss")
axes[1].set_title("Training loss")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/ann_vs_snn.png", dpi=150)
plt.show()

## Optional: TEBN vs plain batch norm

Trains the same SNN with ordinary batch norm to see how much TEBN actually helps. This doubles the SNN training time.

In [ ]:
RUN_TEBN_COMPARISON = False

if RUN_TEBN_COMPARISON:
    torch.manual_seed(42)
    snn_plain = SNN(use_tebn=False)
    plain_history, plain_stats = train_model(snn_plain, "snn_plain_bn")

    print(f"SNN + plain BN: {plain_stats['best_test_acc']:.2f}%")
    print(f"SNN + TEBN:     {snn_stats['best_test_acc']:.2f}%")